<h1>File Code Tanpa MODIS</h1>

<h2>1. Setup & Load Data</h2>

In [59]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.impute import KNNImputer

In [60]:
PROCESSED_DIR = "../data/processed"
MERGED_PATH = os.path.join(PROCESSED_DIR, "air_quality_merged_clean.csv")

df = pd.read_csv(MERGED_PATH, parse_dates = ["date"])
print("Loaded Shape : ", df.shape)

df.head()

Loaded Shape :  (8125, 23)


,id,data_period,date,station,PM10,PM25,SO2,CO,O3,NO2,...,s5p_co,s5p_no2,s5p_o3,s5p_so2,modis_system:index,modis_MODIS_AOD_047,modis_.geo,viirs_system:index,viirs_VIIRS_NTL,viirs_.geo
0,1,202301,2023-01-01,bundaran hotel indonesia,44,55,47,10,24,9,...,NaN,0.000075,0.120394,NaN,NaN,NaN,NaN,0_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,2,202301,2023-01-02,bundaran hotel indonesia,32,43,52,9,24,8,...,NaN,0.000097,0.119363,NaN,NaN,NaN,NaN,1_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,3,202301,2023-01-03,bundaran hotel indonesia,31,35,49,9,12,7,...,NaN,NaN,0.120255,NaN,NaN,NaN,NaN,2_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,4,202301,2023-01-04,bundaran hotel indonesia,30,47,53,11,15,9,...,NaN,0.000140,0.118994,NaN,NaN,NaN,NaN,3_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,5,202301,2023-01-05,bundaran hotel indonesia,38,50,50,13,26,11,...,0.032939,0.000123,0.119520,0.00017,NaN,NaN,NaN,4_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


* <b>NOTE : </b> di hasil merge masih ada modis, ketika habis ini kita drop dan hilangkan dari feature collection

In [61]:
df = df.drop(
    columns=["modis_system:index", "modis_MODIS_AOD_047", "modis_.geo"]
)


In [62]:
df.head()

,id,data_period,date,station,PM10,PM25,SO2,CO,O3,NO2,max,critical_pol,category,s5p_co,s5p_no2,s5p_o3,s5p_so2,viirs_system:index,viirs_VIIRS_NTL,viirs_.geo
0,1,202301,2023-01-01,bundaran hotel indonesia,44,55,47,10,24,9,55,PM25,MEDIUM,NaN,0.000075,0.120394,NaN,0_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,2,202301,2023-01-02,bundaran hotel indonesia,32,43,52,9,24,8,52,SO2,MEDIUM,NaN,0.000097,0.119363,NaN,1_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,3,202301,2023-01-03,bundaran hotel indonesia,31,35,49,9,12,7,49,SO2,GOOD,NaN,NaN,0.120255,NaN,2_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,4,202301,2023-01-04,bundaran hotel indonesia,30,47,53,11,15,9,53,SO2,MEDIUM,NaN,0.000140,0.118994,NaN,3_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,5,202301,2023-01-05,bundaran hotel indonesia,38,50,50,13,26,11,50,PM25,GOOD,0.032939,0.000123,0.119520,0.00017,4_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


In [22]:
df.shape

(8125, 20)

In [23]:
print("Date min/max:", df["date"].min(), "->", df["date"].max())
print("Unique stations:", df["station"].nunique())
print("Unique categories:", df["category"].unique())
print("Total missing (%):", round(df.isna().mean().mean() * 100, 2))


Date min/max: 2023-01-01 00:00:00 -> 2025-10-31 00:00:00
Unique stations: 8
Unique categories: ['MEDIUM' 'GOOD' 'TIDAK ADA DATA' 'UNHEALTHY' 'VERY_UNHEALTHY']
Total missing (%): 6.01


<h2>2. Feature Selection</h2>
<h4>2.1 Definisikan Kolom yang dipakai</h4>
<b><p>* NOTE : disini kita tidak akan pakai MODIS lagi dikarenakan missing nya yang tinggi, silahkan melihat di 03_preprocessing yang memiliki data modis</p></b>

In [24]:
label_col = "category"

ground_cols =["PM10", "PM25", "SO2", "CO", "O3", "NO2", "max"]
s5p_cols = ["s5p_co", "s5p_no2", "s5p_o3", "s5p_so2"]
viirs_cols = ["viirs_VIIRS_NTL"]

key_cols = ["date", "station"]

model_feature_cols = s5p_cols + viirs_cols

keep_cols = key_cols + [label_col] + ground_cols + model_feature_cols
keep_cols


['date',
 'station',
 'category',
 'PM10',
 'PM25',
 'SO2',
 'CO',
 'O3',
 'NO2',
 'max',
 's5p_co',
 's5p_no2',
 's5p_o3',
 's5p_so2',
 'viirs_VIIRS_NTL']

<h4>2.2 Subset Dataset (Drop kolom lain)</h4>

In [25]:
df_sel = df[keep_cols].copy()

print("Before:", df.shape)
print("After :", df_sel.shape)

df_sel.head()


Before: (8125, 20)
After : (8125, 15)


,date,station,category,PM10,PM25,SO2,CO,O3,NO2,max,s5p_co,s5p_no2,s5p_o3,s5p_so2,viirs_VIIRS_NTL
0,2023-01-01,bundaran hotel indonesia,MEDIUM,44,55,47,10,24,9,55,NaN,0.000075,0.120394,NaN,74.275642
1,2023-01-02,bundaran hotel indonesia,MEDIUM,32,43,52,9,24,8,52,NaN,0.000097,0.119363,NaN,74.275642
2,2023-01-03,bundaran hotel indonesia,GOOD,31,35,49,9,12,7,49,NaN,NaN,0.120255,NaN,74.275642
3,2023-01-04,bundaran hotel indonesia,MEDIUM,30,47,53,11,15,9,53,NaN,0.000140,0.118994,NaN,74.275642
4,2023-01-05,bundaran hotel indonesia,GOOD,38,50,50,13,26,11,50,0.032939,0.000123,0.119520,0.00017,74.275642


<h4>2.3 Check Missing Value</h4>

In [26]:
feature_cols = model_feature_cols

missing_table = (
    df_sel[feature_cols]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending = False)
    .rename("Missing %")
    .reset_index()
    .rename(columns = {"index" : "Feature"})
)

missing_table

,Feature,Missing %
0,s5p_so2,51.692308
1,s5p_co,37.735385
2,s5p_no2,12.012308
3,s5p_o3,4.356923
4,viirs_VIIRS_NTL,4.024615


<h2>3. Label Processing</h2>
<h4>3.1 Check Distribusi Label (Before)</h4>

In [27]:
label_before = (
    df_sel["category"]
    .value_counts(dropna = False)
    .rename("count")
    .reset_index()
    .rename(columns = {"index" : "category"})
)

label_before["percentage"] = (
    label_before["count"] / label_before["count"].sum() * 100
)

label_before

,category,count,percentage
0,MEDIUM,3323,40.898462
1,UNHEALTHY,2461,30.289231
2,GOOD,1958,24.098462
3,TIDAK ADA DATA,380,4.676923
4,VERY_UNHEALTHY,3,0.036923


<h4>3.2 Drop Kategori Tidak Valid</h4>
<p>Drop "TIDAK ADA DATA"</p>

In [28]:
before_rows = df_sel.shape[0]

df_sel = df_sel[df_sel["category"] != "TIDAK ADA DATA"].copy()

after_rows = df_sel.shape[0]
print(f"Dropped rows (TIDAK ADA DATA) : {before_rows - after_rows}")

Dropped rows (TIDAK ADA DATA) : 380


<h4>3.3 Ubah VERY_UNHEALTHY -> UNHEALTHY</h4>
<p>Karena data VERY_UNHEALTHY sangat dikit yaitu, count -> 3</p>

In [29]:
df_sel["category"] = df_sel["category"].replace({
    "VERY_UNHEALTHY" : "UNHEALTHY"
})

<h4>3.4 Validasi Distribusi Label (After)</h4>

In [30]:
label_after = (
    df_sel["category"]
    .value_counts()
    .rename("count")
    .reset_index()
    .rename(columns = {"index" : "category"})
)

label_after["percentage"] = (
    label_after["count"] / label_after["count"].sum() * 100
)

label_after

,category,count,percentage
0,MEDIUM,3323,42.905100
1,UNHEALTHY,2464,31.814074
2,GOOD,1958,25.280826


In [31]:
print("Unique label : ", sorted(df_sel["category"].unique()))
print("Total rows after label procesing : ", df_sel.shape[0])

Unique label :  ['GOOD', 'MEDIUM', 'UNHEALTHY']
Total rows after label procesing :  7745


<h2>4. Split Feature & Label</h2>
<h4>4.1 DEfinisikan Kolom Fitur & Label</h4>

In [32]:
y = df_sel["category"].copy()

feature_cols = (
    s5p_cols +
    viirs_cols
)

X = df_sel[feature_cols].copy()

print("X shape : ", X.shape)
print("y shape : ", y.shape)

X shape :  (7745, 5)
y shape :  (7745,)


<h4>4.2 Validasi Tipe Data Fitur</h4>

In [33]:
sat_cols = s5p_cols + viirs_cols
for c in sat_cols :
    if c in X.columns :
        X[c] = pd.to_numeric(X[c], errors="coerce")

print(X.dtypes)

s5p_co             float64
s5p_no2            float64
s5p_o3             float64
s5p_so2            float64
viirs_VIIRS_NTL    float64
dtype: object


In [34]:
missing_before = (
    X.isna()
     .mean()
     .mul(100)
     .sort_values(ascending=False)
     .rename("missing %")
     .reset_index()
     .rename(columns={"index": "feature"})
)

missing_before


,feature,missing %
0,s5p_so2,51.723693
1,s5p_co,38.140736
2,s5p_no2,12.149774
3,s5p_o3,4.286637
4,viirs_VIIRS_NTL,4.144609


In [35]:
print("Jumlah fitur :", X.shape[1])
print("Jumlah sampel :", X.shape[0])
print("Distribusi label :\n", y.value_counts())


Jumlah fitur : 5
Jumlah sampel : 7745
Distribusi label :
 category
MEDIUM       3323
UNHEALTHY    2464
GOOD         1958
Name: count, dtype: int64


<h2>5. Train Test Split</h2>

In [36]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

print("X_train_raw:", X_train_raw.shape)
print("X_test_raw :", X_test_raw.shape)
print("y_train_raw:", y_train_raw.shape)
print("y_test_raw :", y_test_raw.shape)

print("\nDistribusi label (train):")
print(y_train_raw.value_counts(normalize=True).mul(100).round(2))

print("\nDistribusi label (test):")
print(y_test_raw.value_counts(normalize=True).mul(100).round(2))

X_train_raw: (6196, 5)
X_test_raw : (1549, 5)
y_train_raw: (6196,)
y_test_raw : (1549,)

Distribusi label (train):
category
MEDIUM       42.91
UNHEALTHY    31.81
GOOD         25.27
Name: proportion, dtype: float64

Distribusi label (test):
category
MEDIUM       42.87
UNHEALTHY    31.83
GOOD         25.31
Name: proportion, dtype: float64


<h2>6. Handling Missing Value (KNNImputer)</h2>

In [37]:
imputer = KNNImputer(
    n_neighbors=5,
    weights="distance"
)

# fit pada train lalu transform train & tes 
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train_raw),
    columns=X_train_raw.columns,
    index=X_train_raw.index
)

X_test_imp = pd.DataFrame(
    imputer.transform(X_test_raw),
    columns=X_test_raw.columns,
    index=X_test_raw.index
)

print("X_train_imp:", X_train_imp.shape)
print("X_test_imp :", X_test_imp.shape)

# Validasi missing
print("\nMissing % (train) setelah imputasi:")
print((X_train_imp.isna().mean() * 100).round(4).sort_values(ascending=False))

print("\nMissing % (test) setelah imputasi:")
print((X_test_imp.isna().mean() * 100).round(4).sort_values(ascending=False))


X_train_imp: (6196, 5)
X_test_imp : (1549, 5)

Missing % (train) setelah imputasi:
s5p_co             0.0
s5p_no2            0.0
s5p_o3             0.0
s5p_so2            0.0
viirs_VIIRS_NTL    0.0
dtype: float64

Missing % (test) setelah imputasi:
s5p_co             0.0
s5p_no2            0.0
s5p_o3             0.0
s5p_so2            0.0
viirs_VIIRS_NTL    0.0
dtype: float64


<h2>7. Scaling</h2>

In [38]:
scaler = RobustScaler()

# fit pada Train lalu Transform train dan test
X_train_proc = pd.DataFrame(
    scaler.fit_transform(X_train_imp),
    columns=X_train_imp.columns,
    index=X_train_imp.index
)

X_test_proc = pd.DataFrame(
    scaler.transform(X_test_imp),
    columns=X_test_imp.columns,
    index=X_test_imp.index
)

print("X_train_proc:", X_train_proc.shape)
print("X_test_proc :", X_test_proc.shape)

# cek statistik setelah scaling
display(X_train_proc.describe().T[["mean", "std", "min", "max"]])


X_train_proc: (6196, 5)
X_test_proc : (1549, 5)


,mean,std,min,max
s5p_co,0.138038,0.888240,-2.033683,4.203410
s5p_no2,0.194945,0.838533,-1.151807,5.594992
s5p_o3,-0.008462,0.820896,-2.383932,2.419612
s5p_so2,0.049788,1.078075,-4.295478,10.764361
viirs_VIIRS_NTL,0.188587,0.834973,-1.878072,8.391232


<h2>8. Encoding Label</h2>

In [39]:
# Mapping Label
label_mapping = {
    "GOOD" : 0,
    "MEDIUM" : 1,
    "UNHEALTHY" : 2
}

<h4>8.1 Encoding Label Train & Test</h4>

In [40]:
y_train_enc = y_train_raw.map(label_mapping)
y_test_enc = y_test_raw.map(label_mapping)

# Validasi
print("y_train_enc shape:", y_train_enc.shape)
print("y_test_enc shape :", y_test_enc.shape)

print("\nDistribusi label train (encoded):")
print(y_train_enc.value_counts().sort_index())

print("\nDistribusi label test (encoded):")
print(y_test_enc.value_counts().sort_index())


y_train_enc shape: (6196,)
y_test_enc shape : (1549,)

Distribusi label train (encoded):
category
0    1566
1    2659
2    1971
Name: count, dtype: int64

Distribusi label test (encoded):
category
0    392
1    664
2    493
Name: count, dtype: int64


<h2>9. Save Output</h2>

In [52]:
#direktori
PREPROCESS_DIR = "../data/processed/preprocessing/no_modis"
MODEL_DIR = "../models/no_model_modis"

<h4>9.1 Save Features</h4>

In [53]:
X_train_path = os.path.join(PREPROCESS_DIR, "X_train_processed_no_modis.csv")
X_test_path  = os.path.join(PREPROCESS_DIR, "X_test_processed_no_modis.csv")

X_train_proc.to_csv(X_train_path, index=False)
X_test_proc.to_csv(X_test_path, index=False)

print(f"X_train saved to: {X_train_path}")
print(f"X_test saved to : {X_test_path}")

X_train saved to: ../data/processed/preprocessing/no_modis\X_train_processed_no_modis.csv
X_test saved to : ../data/processed/preprocessing/no_modis\X_test_processed_no_modis.csv


<h4>9.2 Save Labels</h4>

In [54]:
y_train_path = os.path.join(PREPROCESS_DIR, "y_train_encoded_no_modis.csv")
y_test_path  = os.path.join(PREPROCESS_DIR, "y_test_encoded_no_modis.csv")

y_train_enc.to_csv(y_train_path, index=False)
y_test_enc.to_csv(y_test_path, index=False)

print(f"y_train saved to: {y_train_path}")
print(f"y_test saved to : {y_test_path}")

y_train saved to: ../data/processed/preprocessing/no_modis\y_train_encoded_no_modis.csv
y_test saved to : ../data/processed/preprocessing/no_modis\y_test_encoded_no_modis.csv


<h4>9.3 Save Imputer & scaler</h4>

In [51]:
imputer_path = os.path.join(PREPROCESS_DIR, "knn_imputer_no_modis.pkl")
scaler_path  = os.path.join(PREPROCESS_DIR, "robust_scaler_no_modis.pkl")

joblib.dump(imputer, imputer_path)
joblib.dump(scaler, scaler_path)

print(f"Imputer saved to: {imputer_path}")
print(f"Scaler saved to : {scaler_path}")

Imputer saved to: ../data/processed/preprocessing/no_modis\knn_imputer_no_modis.pkl
Scaler saved to : ../data/processed/preprocessing/no_modis\robust_scaler_no_modis.pkl
